In [0]:
-- ================= BRONZE =================
COMMENT ON TABLE workspace.default.bronze_nyt_articles IS
'Camada Bronze: artigos do NYT extraídos do MySQL local (nyt_db.nyt_articles), exportados via CSV. Dado bruto, sem tratamento.';

ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN web_url COMMENT 'URL do artigo no NYT (chave natural/PK na origem)';
ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN headline COMMENT 'Título do artigo';
ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN snippet COMMENT 'Trecho curto de abertura do artigo';
ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN abstract COMMENT 'Resumo/abstract do artigo';
ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN published_date COMMENT 'Data/hora de publicação (texto bruto, sem tipagem ainda)';
ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN byline COMMENT 'Autor(es) do artigo';
ALTER TABLE workspace.default.bronze_nyt_articles ALTER COLUMN section COMMENT 'Seção do NYT (ex: U.S., World, Business, Food)';

COMMENT ON TABLE workspace.default.bronze_dxy IS
'Camada Bronze: cotações diárias do índice DXY (ticker DX-Y.NYB), extraídas via yfinance. Dado bruto.';

COMMENT ON TABLE workspace.default.bronze_usdbrl IS
'Camada Bronze: cotações diárias do câmbio USD/BRL (ticker BRL=X), extraídas via yfinance. Dado bruto.';

-- ================= SILVER =================
COMMENT ON TABLE workspace.default.silver_nyt_articles IS
'Camada Silver: artigos do NYT limpos (published_date tipado com try_to_timestamp, 314 registros com data corrompida removidos) e enriquecidos com sentimento VADER (coluna sentiment, -1 a 1) sobre headline+abstract+snippet.';

ALTER TABLE workspace.default.silver_nyt_articles ALTER COLUMN published_date COMMENT 'Timestamp de publicação, já tipado e validado (try_to_timestamp)';
ALTER TABLE workspace.default.silver_nyt_articles ALTER COLUMN sentiment COMMENT 'Score de sentimento VADER (compound), de -1 (muito negativo) a 1 (muito positivo)';

COMMENT ON TABLE workspace.default.silver_dxy IS
'Camada Silver: cotações do DXY com schema padronizado (Date, Open, High, Low, Close, Volume), sem sufixos de ticker.';

COMMENT ON TABLE workspace.default.silver_usdbrl IS
'Camada Silver: cotações do USD/BRL com schema padronizado (Date, Open, High, Low, Close, Volume), sem sufixos de ticker.';

-- ================= GOLD =================
COMMENT ON TABLE workspace.default.gold_analise_diaria IS
'Camada Gold: uma linha por dia, com interseção entre dias com notícias do NYT e dias com cotação de câmbio disponível (401 dias). Base para responder P1-P4 (volume/sentimento de notícias vs. DXY/USDBRL).';

ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN dia COMMENT 'Data (granularidade diária), chave da tabela';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN total_artigos COMMENT 'Quantidade de artigos do NYT publicados no dia';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN sentimento_medio COMMENT 'Média do score de sentimento VADER dos artigos do dia';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN dxy_close COMMENT 'Fechamento do índice DXY no dia';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN dxy_return COMMENT 'Retorno percentual do DXY em relação ao fechamento anterior';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN usdbrl_close COMMENT 'Fechamento do câmbio USD/BRL no dia';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN usdbrl_return COMMENT 'Retorno percentual do USD/BRL em relação ao fechamento anterior';
ALTER TABLE workspace.default.gold_analise_diaria ALTER COLUMN sentimento_lag1 COMMENT 'Sentimento médio do dia anterior (usado para testar antecipação/lag, P3)';